
# Rule-Based Padang Food Classification V2 (Handcrafted, No ML/DL)

Notebook ini berisi versi **V2** dari pipeline klasifikasi makanan Padang berbasis **rule-based murni (handcrafted)**, dengan perbaikan pada:

1. **Segmentasi ROI**
2. **Fitur rule-based** (lebih kaya, tidak terlalu kasar untuk kelas mirip)
3. **Aturan klasifikasi** (tetap rule-based, tapi lebih terstruktur)
4. **Evaluasi** (akurasi, macro-F1, report per kelas, confusion matrix)
5. **Penanganan mismatch nama kelas** (hanya 9 kelas seperti di dataset)
6. **Perbaikan bug fitur (GLCM & casting float)** untuk menghindari error
   `The truth value of an array with more than one element is ambiguous`.

Silakan sesuaikan `dataset_dir` dengan lokasi dataset kamu.


In [1]:

import os
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Agar plot tampil di notebook
%matplotlib inline

# Ukuran gambar target (sisi terpendek)
img_size = 256

# Ganti path ini sesuai lokasi dataset kamu
# Struktur yang diasumsikan:
# dataset_dir/
#   ayam_goreng/
#       img1.jpg, img2.jpg, ...
#   rendang/
#       ...
dataset_dir = 'dataset_padang_food'


## Utilitas dasar: baca & tampilkan gambar, list path dataset

In [2]:

def imread_resize(path, target_short=256):
    """
    Baca gambar BGR (cv2) dan resize sehingga sisi terpendek = target_short,
    menjaga rasio aspek.
    """
    img = cv2.imread(path)
    if img is None:
        raise IOError(f'Cannot read image: {path}')
    h, w = img.shape[:2]
    if min(h, w) == target_short:
        return img
    scale = target_short / float(min(h, w))
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return img_resized

def show_imgs(imgs, cols=3, titles=None, figsize=(12, 6)):
    """Tampilkan beberapa gambar dalam grid (BGR atau GRAY)."""
    n = len(imgs)
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=figsize)
    for i, img in enumerate(imgs):
        plt.subplot(rows, cols, i+1)
        if img.ndim == 2:
            plt.imshow(img, cmap='gray')
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if titles is not None and i < len(titles):
            plt.title(titles[i])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

def list_image_paths(root_dir):
    """
    Mengembalikan:
      - classes: daftar nama folder (kelas)
      - all_paths: list tuple (path_img, label_folder)
    """
    classes = []
    all_paths = []

    for name in sorted(os.listdir(root_dir)):
        d = os.path.join(root_dir, name)
        if not os.path.isdir(d):
            continue
        classes.append(name)
        for fname in sorted(os.listdir(d)):
            fpath = os.path.join(d, fname)
            if os.path.isfile(fpath):
                all_paths.append((fpath, name))
    return classes, all_paths


## Segmentasi ROI V2 (gray-world + HSV + Otsu + anti-piring)

In [3]:

def gray_world(img_bgr):
    """White balance sederhana metode gray-world."""
    img = img_bgr.astype(np.float32)
    mean_bgr = img.reshape(-1, 3).mean(axis=0)
    gray_val = float(np.mean(mean_bgr) + 1e-6)
    scale = gray_val / (mean_bgr + 1e-6)
    img *= scale
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

def segment_food_roi_v2(img_bgr, debug=False):
    """
    Segmentasi makanan V2:
    1) Gray-world → HSV
    2) Otsu di S dan V → threshold adaptif
    3) Bersihkan dengan morfologi + ambil komponen terbesar
    4) Jika gagal → center crop (bukan full image)
    """
    h, w = img_bgr.shape[:2]
    img = gray_world(img_bgr)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hch, sch, vch = cv2.split(hsv)

    sch_blur = cv2.GaussianBlur(sch, (5, 5), 0)
    vch_blur = cv2.GaussianBlur(vch, (5, 5), 0)

    _, s_otsu = cv2.threshold(sch_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, v_otsu = cv2.threshold(vch_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    s_thr = max(10, s_otsu - 10)
    v_thr = max(40, v_otsu - 10)

    s_mask = (sch_blur >= s_thr).astype(np.uint8)
    v_mask = (vch_blur >= v_thr).astype(np.uint8)

    mask_sv = (s_mask & v_mask).astype(np.uint8) * 255

    white_mask = ((sch_blur <= 30) & (vch_blur >= 200)).astype(np.uint8) * 255
    mask = cv2.bitwise_and(mask_sv, cv2.bitwise_not(white_mask))

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        ch, cw = int(h * 0.6), int(w * 0.6)
        y0 = (h - ch) // 2
        x0 = (w - cw) // 2
        mask_fallback = np.zeros((h, w), np.uint8)
        mask_fallback[y0:y0 + ch, x0:x0 + cw] = 255
        return mask_fallback

    largest = max(cnts, key=cv2.contourArea)
    mask_final = np.zeros_like(mask)
    cv2.drawContours(mask_final, [largest], -1, 255, thickness=-1)

    if debug:
        show_imgs(
            [img, cv2.cvtColor(mask_final, cv2.COLOR_GRAY2BGR)],
            cols=2,
            titles=['Gray-world BGR', 'Mask Food']
        )

    return mask_final

def get_mask_v2(img):
    return segment_food_roi_v2(img)


## Ekstraksi fitur V2 (warna + tekstur + bentuk) + perbaikan GLCM & casting float

In [4]:

def safe_float(x, default=0.0):
    try:
        if isinstance(x, np.ndarray):
            if x.size == 1:
                x = x.item()
            else:
                x = float(np.mean(x))
        x = float(x)
        if np.isnan(x):
            return default
        return x
    except Exception:
        return default

def compute_shape_features_v2(mask):
    m = (mask > 0).astype(np.uint8)
    total_area = m.size
    area = float(m.sum())
    if area <= 0:
        return dict(
            area_ratio=0.0, aspect_ratio=1.0, extent=0.0,
            roundness=0.0, convexity=0.0, num_blobs=0
        )
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    num_blobs = len(cnts)
    cnt = max(cnts, key=cv2.contourArea)
    perimeter = cv2.arcLength(cnt, True) + 1e-6
    x, y, w, h = cv2.boundingRect(cnt)
    aspect = w / h if h > 0 else 1.0
    extent = area / float(w * h + 1e-6)
    roundness = 4 * math.pi * cv2.contourArea(cnt) / (perimeter ** 2)
    hull = cv2.convexHull(cnt)
    convexity = cv2.contourArea(cnt) / (cv2.contourArea(hull) + 1e-6)
    area_ratio = area / float(total_area)
    return dict(
        area_ratio=safe_float(area_ratio),
        aspect_ratio=safe_float(aspect),
        extent=safe_float(extent),
        roundness=safe_float(roundness),
        convexity=safe_float(convexity),
        num_blobs=int(num_blobs)
    )

def compute_color_features_v2(img_bgr, mask):
    img = gray_world(img_bgr)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hch, sch, vch = cv2.split(hsv)
    m = (mask > 0)
    if m.sum() == 0:
        m = np.ones_like(hch, dtype=bool)

    h = hch[m].astype(np.float32)
    s = sch[m].astype(np.float32)
    v = vch[m].astype(np.float32)

    h_n = h / 180.0
    s_n = s / 255.0
    v_n = v / 255.0

    bins = np.linspace(0, 1.0, 7)
    hist_h, _ = np.histogram(h_n, bins=bins)
    hist_h = hist_h.astype(np.float32) / (hist_h.sum() + 1e-6)

    yellow_mask = (h_n >= 30/180.0) & (h_n <= 70/180.0)
    brown_mask  = (v_n < 0.5) & (h_n >= 10/180.0) & (h_n <= 50/180.0)

    yellow_ratio = yellow_mask.mean() if yellow_mask.size > 0 else 0.0
    brown_ratio  = brown_mask.mean() if brown_mask.size > 0 else 0.0

    feats = dict(
        h_bin_0=safe_float(hist_h[0]),
        h_bin_1=safe_float(hist_h[1]),
        h_bin_2=safe_float(hist_h[2]),
        h_bin_3=safe_float(hist_h[3]),
        h_bin_4=safe_float(hist_h[4]),
        h_bin_5=safe_float(hist_h[5]),
        sat_mean=safe_float(s_n.mean()),
        sat_std=safe_float(s_n.std()),
        val_mean=safe_float(v_n.mean()),
        val_std=safe_float(v_n.std()),
        yellow_ratio=safe_float(yellow_ratio),
        brown_ratio=safe_float(brown_ratio),
        bright_ratio=safe_float((v_n > 0.8).mean()),
        dark_ratio=safe_float((v_n < 0.3).mean())
    )
    return feats

def compute_texture_features_v2(img_bgr, mask):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    m = (mask > 0)
    if m.sum() == 0:
        m = np.ones_like(gray, dtype=bool)
    roi = gray.copy()
    roi[~m] = 0

    P = 8
    R = 1
    lbp = local_binary_pattern(roi, P, R, method='uniform')
    lbp_vals = lbp[m]
    hist_lbp, _ = np.histogram(lbp_vals, bins=np.arange(0, P+3), density=True)
    lbp_uniform_ratio = float(hist_lbp[:-1].sum())
    lbp_nonuniform_ratio = float(hist_lbp[-1])

    roi8 = cv2.normalize(roi, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    try:
        glcm = graycomatrix(roi8, distances=[1], angles=[0], levels=256,
                            symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]
    except Exception:
        contrast, homogeneity, energy = 0.0, 0.0, 0.0

    return dict(
        lbp_uniform=safe_float(lbp_uniform_ratio),
        lbp_nonuniform=safe_float(lbp_nonuniform_ratio),
        glcm_contrast=safe_float(contrast),
        glcm_homogeneity=safe_float(homogeneity),
        glcm_energy=safe_float(energy)
    )

def extract_rule_descriptors_v2(img_bgr, mask):
    color = compute_color_features_v2(img_bgr, mask)
    texture = compute_texture_features_v2(img_bgr, mask)
    shape = compute_shape_features_v2(mask)
    feats = {}
    feats.update(color)
    feats.update(texture)
    feats.update(shape)

    for k, v in list(feats.items()):
        if isinstance(v, np.ndarray):
            feats[k] = safe_float(v)
        elif v is None:
            feats[k] = 0.0
        else:
            if k == 'num_blobs':
                feats[k] = int(v)
            else:
                feats[k] = safe_float(v)
    return feats


## Build dataset fitur V2

In [5]:

def build_rule_dataset_v2(all_paths):
    data = []
    for p, yc in all_paths:
        img = imread_resize(p, target_short=img_size)
        mask = get_mask_v2(img)
        feats = extract_rule_descriptors_v2(img, mask)
        data.append({'path': p, 'label': yc, 'features': feats})
    return data

try:
    classes_raw, all_paths = list_image_paths(dataset_dir)
    print("Ditemukan kelas (folder):", classes_raw)
    print("Total gambar:", len(all_paths))
except Exception as e:
    print("Gagal membaca dataset_dir, pastikan path benar:", e)


Ditemukan kelas (folder): ['ayam_goreng', 'ayam_pop', 'daging_rendang', 'dendeng_batokok', 'gulai_ikan', 'gulai_tambusu', 'gulai_tunjang', 'telur_balado', 'telur_dadar']
Total gambar: 783


## Normalisasi nama kelas (canonical label & alias) - 9 kelas dataset

In [6]:

CANONICAL_LABELS = [
    'ayam_goreng', 'ayam_pop',
    'daging_rendang', 'dendeng_batokok',
    'gulai_ikan', 'gulai_tambusu', 'gulai_tunjang',
    'telur_balado', 'telur_dadar'
]

LABEL_ALIASES = {
    'rendang': 'daging_rendang',
    'daging_rendang': 'daging_rendang',
    'dendeng': 'dendeng_batokok',
    'dendeng_batokok': 'dendeng_batokok',
    'telur_bulat_balado': 'telur_balado',
    'telur_bulat': 'telur_balado',
}

def canonicalize_label(y):
    return LABEL_ALIASES.get(y, y)


## Helper: ringkasan statistik fitur per kelas

In [7]:

def summarize_feature_stats(dataset, feature_names):
    rows = []
    for d in dataset:
        row = {'label': canonicalize_label(d['label'])}
        for fn in feature_names:
            row[fn] = d['features'].get(fn, np.nan)
        rows.append(row)
    df = pd.DataFrame(rows)
    stats = df.groupby('label')[feature_names].agg(['mean', 'std', 'min', 'max'])
    return stats


## Rule-based classifier V2 (family → subkelas, 9 kelas)

In [8]:

def family_routing(feats):
    yel = feats['yellow_ratio']
    brn = feats['brown_ratio']
    bright = feats['bright_ratio']
    dark = feats['dark_ratio']
    roundness = feats['roundness']
    area_ratio = feats['area_ratio']

    if (roundness > 0.45) and (area_ratio > 0.08) and (dark < 0.4):
        return 'telur'

    if (yel > 0.25) and (bright > 0.2) and (dark < 0.5):
        return 'gulai'

    return 'kering'

def classify_in_telur(feats):
    brn = feats['brown_ratio']
    dark = feats['dark_ratio']
    if (dark > 0.35) and (brn > 0.2):
        return 'telur_balado'
    return 'telur_dadar'

def classify_in_gulai(feats):
    yel = feats['yellow_ratio']
    dark = feats['dark_ratio']
    contrast = feats['glcm_contrast']
    area_ratio = feats['area_ratio']
    aspect = feats['aspect_ratio']
    num_blobs = feats['num_blobs']

    if (area_ratio > 0.25) and (contrast > 80):
        return 'gulai_tunjang'

    if (num_blobs <= 2) and (aspect > 1.4):
        return 'gulai_ikan'

    return 'gulai_tambusu'

def classify_in_kering(feats):
    brn = feats['brown_ratio']
    dark = feats['dark_ratio']
    contrast = feats['glcm_contrast']
    hom = feats['glcm_homogeneity']
    area_ratio = feats['area_ratio']

    if (dark > 0.45) and (brn > 0.25):
        if (area_ratio < 0.12) and (contrast > 120):
            return 'dendeng_batokok'
        else:
            return 'daging_rendang'

    if (dark < 0.35) and (hom > 0.4):
        return 'ayam_pop'
    else:
        return 'ayam_goreng'

def classify_rule_v2(feats):
    fam = family_routing(feats)
    if fam == 'telur':
        return classify_in_telur(feats)
    elif fam == 'gulai':
        return classify_in_gulai(feats)
    else:
        return classify_in_kering(feats)


## Evaluasi: akurasi, macro-F1, report per kelas, confusion matrix

In [9]:

def evaluate_rule_based_v2(dataset, classifier_fn=classify_rule_v2):
    y_true = []
    y_pred = []

    for d in dataset:
        yt = canonicalize_label(d['label'])
        yp = classifier_fn(d['features'])
        y_true.append(yt)
        y_pred.append(yp)

    labels = sorted(list(set(y_true) | set(y_pred)))

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)

    print("Accuracy :", acc)
    print("Macro-F1 :", macro_f1)
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize='true')
    print("\nConfusion matrix (normalized by true class):")
    print("labels order:", labels)
    print(cm)

    return {
        'accuracy': acc,
        'macro_f1': macro_f1,
        'labels': labels,
        'cm': cm,
        'y_true': y_true,
        'y_pred': y_pred
    }


## Contoh penggunaan end-to-end

In [10]:

# 1. Build dataset fitur V2
try:
    classes_raw, all_paths = list_image_paths(dataset_dir)
    print("Kelas (folder) ditemukan:", classes_raw)
    print("Total gambar:", len(all_paths))

    rule_dataset_v2 = build_rule_dataset_v2(all_paths)
    print("Selesai ekstrak fitur. Contoh item:")
    print(rule_dataset_v2[0]['label'])
    print(list(rule_dataset_v2[0]['features'].keys())[:10])
except Exception as e:
    print("Gagal membangun dataset fitur. Cek dataset_dir:", e)


Kelas (folder) ditemukan: ['ayam_goreng', 'ayam_pop', 'daging_rendang', 'dendeng_batokok', 'gulai_ikan', 'gulai_tambusu', 'gulai_tunjang', 'telur_balado', 'telur_dadar']
Total gambar: 783
Gagal membangun dataset fitur. Cek dataset_dir: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


In [11]:

# 2. Opsional: statistik fitur
if 'rule_dataset_v2' in globals():
    key_feats = [
        'yellow_ratio', 'brown_ratio', 'bright_ratio', 'dark_ratio',
        'glcm_contrast', 'glcm_homogeneity', 'roundness', 'area_ratio'
    ]
    stats = summarize_feature_stats(rule_dataset_v2, key_feats)
    display(stats)
else:
    print("rule_dataset_v2 belum tersedia.")


rule_dataset_v2 belum tersedia.


In [ ]:

# 3. Evaluasi rule-based V2
if 'rule_dataset_v2' in globals():
    metrics_v2 = evaluate_rule_based_v2(rule_dataset_v2)
else:
    print("rule_dataset_v2 belum tersedia.")


In [ ]:

# 4. (Opsional) cek label 'hantu'
if 'metrics_v2' in globals():
    y_true = metrics_v2['y_true']
    y_pred = metrics_v2['y_pred']
    print("Label y_true:", sorted(set(y_true)))
    print("Label y_pred:", sorted(set(y_pred)))
    print("Label di prediksi tapi tidak ada di ground-truth:",
          sorted(set(y_pred) - set(y_true)))
else:
    print("metrics_v2 belum tersedia.")
